# 제출용 파이프라인 — 전처리부터 학습·추론·결과생성까지 자동 실행

이 노트북 하나로 **데이터 전처리 → 피처 생성 → 모델 학습 → 평가 → 추론 데모**까지
처음부터 끝까지 자동으로 실행된다(Run All). 피처를 어떻게 탐색했는지의 과정(EDA)은
담지 않고, 이미 확정된 최종 피처·모델만 재현한다 — 탐색 과정과 근거는
`docs/01_eda_findings.md`, `docs/02_feature_candidates.md`,
`docs/03_modeling_strategy.md`, `docs/04_methodology.md`와 `notebooks/01~06`에
기록돼 있다.

환경: conda `my_base`(pandas 2.2.3 / scikit-learn 1.6.1). 데이터 경로는
`KAMP_DATA_DIR` 환경변수로 지정한다(미지정 시 기본 경로 사용).

## 0. 환경 설정

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LinearRegression
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

DATA_DIR = Path(os.environ.get('KAMP_DATA_DIR', '/mnt/d/data/kamp_ai/press_anomaly_dataset'))
GAP_THR = .15        # 사이클(세그먼트) 경계 판정 기준(초)
MIN_N = 10           # 메인 모델 적용 최소 세그먼트 길이
COVERAGE = .999      # 정상 train 분위수 기준 임계(= 허용 오경보율 0.1%)
SIGNALS = ['AI0_Vibration', 'AI1_Vibration', 'AI2_Current']
RANDOM_STATE = 0
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 180)
print('DATA_DIR =', DATA_DIR)

DATA_DIR = /mnt/d/data/kamp_ai/press_anomaly_dataset


## 1. 데이터 로드·전처리

완전중복 제거 → 100ms 간격 타임스탬프에서 gap(>0.15초) 기준으로 세그먼트(cycle)
복원 → 세그먼트 단위 원시 RMS 계산 → 정상 세그먼트를 시간순 60/20/20으로 분할한다.

In [2]:
SEG, TABLE, audit = {}, {}, []
for name, filename in [('normal', 'press_data_normal.csv'), ('outlier', 'outlier_data.csv')]:
    raw = pd.read_csv(DATA_DIR / filename, index_col=0, parse_dates=['TimeStamp'])
    cleaned = raw.drop_duplicates().copy()
    assert not cleaned.TimeStamp.duplicated().any(), '동일 timestamp의 서로 다른 값 확인 필요'
    assert cleaned.TimeStamp.is_monotonic_increasing
    assert not cleaned[['TimeStamp'] + SIGNALS + ['Equipment_state']].isna().any().any()
    assert cleaned.Equipment_state.eq(0 if name == 'normal' else 1).all()
    cleaned['seg_id'] = (cleaned.TimeStamp.diff().dt.total_seconds() > GAP_THR).cumsum()
    SEG[name] = cleaned
    rows = []
    for sid, g in cleaned.groupby('seg_id'):
        row = dict(seg_id=sid, n=len(g), t_start=g.TimeStamp.iloc[0], t_end=g.TimeStamp.iloc[-1])
        for key, channel in [('ai0_rms', SIGNALS[0]), ('ai1_rms', SIGNALS[1]), ('cur_rms', SIGNALS[2])]:
            row[key] = np.sqrt(np.mean(g[channel].to_numpy(float)**2))
        rows.append(row)
    TABLE[name] = pd.DataFrame(rows).sort_values('t_start').reset_index(drop=True)
    audit.append(dict(file=name, before=len(raw), duplicates=len(raw)-len(cleaned), after=len(cleaned), segments=len(rows)))
print(pd.DataFrame(audit).to_string(index=False))

normal = TABLE['normal'].copy()
a, b = int(len(normal)*.6), int(len(normal)*.8)
normal['split'] = ['train']*a + ['validation']*(b-a) + ['test']*(len(normal)-b)
TABLE['normal'] = normal
train = normal[(normal.split == 'train') & (normal.n >= MIN_N)].copy()
assert len(train) > 0
print(normal.groupby('split', sort=False).agg(total=('n', 'size'), eligible=('n', lambda x: (x>=MIN_N).sum())).to_string())
print('outlier:', len(TABLE['outlier']), 'eligible(n>=10):', (TABLE['outlier'].n>=MIN_N).sum())

   file  before  duplicates  after  segments
 normal   20000           1  19999       599
outlier     600           0    600        21
            total  eligible
split                      
train         359       313
validation    120       108
test          120       109
outlier: 21 eligible(n>=10): 17


## 2. 피처 엔지니어링(확정된 최종 피처만)

탐색 과정(6개→VIF→3개 축소, n<10 대응 비교실험 등)은 생략하고 **이미 확정된 식**만
계산한다. 근거는 `docs/02_feature_candidates.md`·`docs/04_methodology.md` 참고.

- `cur_ai0_resid`, `cur_ai1_resid`: 정상 train 기준 `전류_rms → 진동_rms` 선형회귀의 잔차.
- `normal_center_nrmse`: 정상 공통 주파수·중심을 고정한 사인 적합 잔차(NRMSE).
- **메인 피처셋**(n≥10): `cur_ai0_resid, cur_ai1_resid, normal_center_nrmse` (3개).
- **폴백 피처셋**(2≤n<10): `cur_ai0_resid, cur_ai1_resid` (2개) — `normal_center_nrmse`는
  짧은 샘플에서 자유도 부족으로 불안정해지므로 제외(근거: 3장 오류분석).

In [3]:
def sine_fit(g, frequency, center=None):
    x = g.AI2_Current.to_numpy(float)
    t = (g.TimeStamp - g.TimeStamp.iloc[0]).dt.total_seconds().to_numpy()
    design = np.column_stack([np.cos(2*np.pi*frequency*t), np.sin(2*np.pi*frequency*t)])
    if center is None:
        design = np.column_stack([design, np.ones(len(x))])
        coef = np.linalg.lstsq(design, x, rcond=None)[0]
        pred = design @ coef
    else:
        coef = np.linalg.lstsq(design, x - center, rcond=None)[0]
        pred = center + design @ coef
    ss = np.sum((x - pred)**2)
    total = np.sum((x - x.mean())**2)
    return ss, (np.sqrt(ss/total) if total > 0 else np.nan), coef

# 정상 공통 주파수·중심 추정(기준 표본: train n>=30)
reference = train[train.n >= 30]
assert len(reference) > 0
grid = np.linspace(1/2.2, 1/.9, 401)
frequencies = []
for sid in reference.seg_id:
    g = SEG['normal'][SEG['normal'].seg_id == sid]
    best = int(np.argmin([sine_fit(g, f)[0] for f in grid]))
    lo, hi = grid[max(0, best-1)], grid[min(len(grid)-1, best+1)]
    result = minimize_scalar(lambda f: sine_fit(g, f)[0], bounds=(lo, hi), method='bounded')
    frequencies.append(min([grid[best], result.x], key=lambda f: sine_fit(g, f)[0]))
f_reference = float(np.median(frequencies))
center = float(np.median([sine_fit(SEG['normal'][SEG['normal'].seg_id == sid], f_reference)[2][-1]
                          for sid in reference.seg_id]))

# 전류-진동 결합 회귀(정상 train 기준)
regression = {key: np.polyfit(train.cur_rms, train[key], 1) for key in ['ai0_rms', 'ai1_rms']}

for name, table in TABLE.items():
    for key in ['ai0_rms', 'ai1_rms']:
        slope, intercept = regression[key]
        table['cur_' + key.split('_')[0] + '_resid'] = table[key] - (slope*table.cur_rms + intercept)
    table['normal_center_nrmse'] = [sine_fit(SEG[name][SEG[name].seg_id == sid], f_reference, center)[1]
                                    for sid in table.seg_id]

FEATURES_MAIN = ['cur_ai0_resid', 'cur_ai1_resid', 'normal_center_nrmse']
FEATURES_FALLBACK = ['cur_ai0_resid', 'cur_ai1_resid']

# 공선성 정상화 확인(sanity check) — 축소 전 6개는 조건수 1e18, 축소 후 3개는 ~2.55
# train은 피처 컬럼 추가 전 복사본이라 다시 만든다(오래된 복사본 재사용 버그 방지)
train = TABLE['normal'].query("split == 'train' and n >= @MIN_N")
Xchk = train[FEATURES_MAIN].to_numpy(float)
print(f'정상 주파수={f_reference:.6f}Hz  중심={center:.6f}  기준표본={len(reference)}개')
print(f'메인 피처셋 공분산 조건수(sanity check) = {np.linalg.cond(np.cov(Xchk.T)):.2f}')

정상 주파수=0.600039Hz  중심=0.555780  기준표본=215개
메인 피처셋 공분산 조건수(sanity check) = 2.55


## 3. 데이터 분할 — 길이(n) 기반 2-track

세그먼트 길이에 따라 세 구간으로 나눈다(근거: 3장 오류분석 — 길이가 오경보·미탐의
핵심 공정조건).

- **n≥10**: 메인 모델(3개 피처) 적용 대상.
- **2≤n<10**: 폴백 모델(2개 피처) 적용 대상.
- **n=1**: `normal_center_nrmse` 계산 불가(NaN) — 판정 보류.

In [4]:
def split_by_length(table):
    main = table[table.n >= MIN_N]
    fallback = table[(table.n >= 2) & (table.n < MIN_N)]
    deferred = table[table.n < 2]
    return main, fallback, deferred

nrm_main, nrm_fb, nrm_defer = split_by_length(TABLE['normal'])
out_main, out_fb, out_defer = split_by_length(TABLE['outlier'])
print(f"정상: n>=10 {len(nrm_main)}개 / n=2~9 {len(nrm_fb)}개 / n=1(보류) {len(nrm_defer)}개")
print(f"이상: n>=10 {len(out_main)}개 / n=2~9 {len(out_fb)}개 / n=1(보류) {len(out_defer)}개")

train_main = nrm_main.query("split == 'train'")
val_main   = nrm_main.query("split == 'validation'")
test_main  = nrm_main.query("split == 'test'")

# 폴백 모델용 train/test(정상 n=2~9 자체를 시간순 80/20으로 분할 — 표본이 작아 validation은 생략)
nrm_fb_sorted = nrm_fb.sort_values('t_start').reset_index(drop=True)
c = int(len(nrm_fb_sorted) * .8)
train_fb = nrm_fb_sorted.iloc[:c]
test_fb  = nrm_fb_sorted.iloc[c:]
print(f"폴백 train(n=2~9) {len(train_fb)}개 / test {len(test_fb)}개")

정상: n>=10 530개 / n=2~9 60개 / n=1(보류) 9개
이상: n>=10 17개 / n=2~9 4개 / n=1(보류) 0개
폴백 train(n=2~9) 48개 / test 12개


## 4. 모델 학습

베이스라인(정적 규칙) 1개 + 비교 모델 3개(Mahalanobis·Isolation Forest·
One-Class SVM) + 길이 폴백 모델 1개를 전부 학습한다. 하이퍼파라미터(One-Class
SVM의 `nu=0.1` 등)는 이미 확정된 값을 그대로 쓴다(탐색 과정은 `05_oneclass.ipynb`
참고).

In [5]:
# --- 4.1 베이스라인: 정적 규칙(04) ---
# 원본 6개 피처 기준 99.9% 분위수 규칙. ratio_ai1_ai0는 베이스라인에만 쓴다.
TABLE['normal']['ratio_ai1_ai0'] = TABLE['normal'].ai1_rms / TABLE['normal'].ai0_rms.replace(0, np.nan)
TABLE['outlier']['ratio_ai1_ai0'] = TABLE['outlier'].ai1_rms / TABLE['outlier'].ai0_rms.replace(0, np.nan)
train_main = TABLE['normal'].query("split == 'train' and n >= @MIN_N")

FEATURES_BASELINE = ['ai0_rms', 'ai1_rms', 'cur_ai0_resid', 'cur_ai1_resid', 'ratio_ai1_ai0', 'normal_center_nrmse']
TWO_SIDED = {'cur_ai0_resid', 'cur_ai1_resid', 'ratio_ai1_ai0'}
baseline_thr = {}
for f in FEATURES_BASELINE:
    v = train_main[f]
    lo = v.quantile((1 - COVERAGE) / 2) if f in TWO_SIDED else -np.inf
    hi = v.quantile(1 - (1 - COVERAGE) / 2 if f in TWO_SIDED else COVERAGE)
    baseline_thr[f] = (lo, hi)

def baseline_predict(table, feats=FEATURES_BASELINE):
    valid = (table.n >= MIN_N) & np.isfinite(table[feats]).all(axis=1)
    alarm = pd.Series(False, index=table.index)
    for f in feats:
        lo, hi = baseline_thr[f]
        alarm |= (table[f] < lo) | (table[f] > hi)
    return valid, alarm & valid

print('베이스라인(정적 규칙 OR, 6개 피처) 학습 완료 — 임계:')
print(pd.DataFrame([dict(feature=f, lower=v[0], upper=v[1]) for f, v in baseline_thr.items()]).to_string(index=False))

베이스라인(정적 규칙 OR, 6개 피처) 학습 완료 — 임계:
            feature     lower    upper
            ai0_rms      -inf 0.137925
            ai1_rms      -inf 0.215677
      cur_ai0_resid -0.027255 0.069598
      cur_ai1_resid -0.034944 0.065428
      ratio_ai1_ai0  0.473615 2.943657
normal_center_nrmse      -inf 0.093495


In [6]:
# --- 4.2 메인 모델: Mahalanobis distance(n>=10, 3개 피처) ---
X_train_main = train_main[FEATURES_MAIN].to_numpy(float)
cov_main = LedoitWolf().fit(X_train_main)
mean_main = X_train_main.mean(axis=0)

def maha_main(X):
    diff = X - mean_main
    return np.sqrt(np.einsum('ij,jk,ik->i', diff, cov_main.precision_, diff))

maha_thr = float(np.quantile(maha_main(X_train_main), COVERAGE))
print(f'[메인] Mahalanobis train 99.9%ile 임계 = {maha_thr:.4f}')

[메인] Mahalanobis train 99.9%ile 임계 = 5.4207


In [7]:
# --- 4.3 비교 모델: Isolation Forest(n>=10, 3개 피처) ---
iso_main = IsolationForest(n_estimators=300, random_state=RANDOM_STATE, contamination='auto').fit(X_train_main)

def iso_score_main(X):
    return -iso_main.decision_function(X)

iso_thr = float(np.quantile(iso_score_main(X_train_main), COVERAGE))
print(f'[비교] IsolationForest train 99.9%ile 임계 = {iso_thr:.4f}')

[비교] IsolationForest train 99.9%ile 임계 = 0.2415


In [8]:
# --- 4.4 비교 모델: One-Class SVM(n>=10, 3개 피처, nu=0.1 확정값) ---
scaler_main = StandardScaler().fit(X_train_main)
ocsvm_main = OneClassSVM(kernel='rbf', gamma='scale', nu=0.1).fit(scaler_main.transform(X_train_main))

def ocsvm_score_main(X):
    return -ocsvm_main.decision_function(scaler_main.transform(X))

ocsvm_thr = float(np.quantile(ocsvm_score_main(X_train_main), COVERAGE))
print(f'[비교] One-Class SVM(nu=0.1) train 99.9%ile 임계 = {ocsvm_thr:.4f}')

MODELS_MAIN = {'mahalanobis': (maha_main, maha_thr), 'isolation_forest': (iso_score_main, iso_thr),
               'one_class_svm': (ocsvm_score_main, ocsvm_thr)}

[비교] One-Class SVM(nu=0.1) train 99.9%ile 임계 = 2.4025


In [9]:
# --- 4.5 폴백 모델: Mahalanobis(2<=n<10, 2개 피처) ---
X_train_fb = train_fb[FEATURES_FALLBACK].to_numpy(float)
cov_fb = LedoitWolf().fit(X_train_fb)
mean_fb = X_train_fb.mean(axis=0)

def maha_fb(X):
    diff = X - mean_fb
    return np.sqrt(np.einsum('ij,jk,ik->i', diff, cov_fb.precision_, diff))

maha_fb_thr = float(np.quantile(maha_fb(X_train_fb), COVERAGE))
print(f'[폴백] Mahalanobis(2개 피처, n=2~9) train 99.9%ile 임계 = {maha_fb_thr:.4f}')

[폴백] Mahalanobis(2개 피처, n=2~9) train 99.9%ile 임계 = 3.7938


## 5. 평가

정상탐지율·오경보율·이상탐지율·미탐률·Balanced Accuracy·F1을 validation/test에서
보고한다. 이상(outlier)은 학습에 쓰지 않고 평가에만 투입한다.

In [10]:
def score_table(table, score_fn, feats, min_n=MIN_N):
    valid = (table.n >= min_n) & np.isfinite(table[feats]).all(axis=1)
    X = table.loc[valid, feats].to_numpy(float)
    s = pd.Series(score_fn(X), index=table.index[valid]) if valid.any() else pd.Series(dtype=float)
    return valid, s

def metrics_row(split_name, rule_name, nt, ot, score_fn, thr, feats, min_n=MIN_N):
    nvalid, ns = score_table(nt, score_fn, feats, min_n)
    ovalid, os_ = score_table(ot, score_fn, feats, min_n)
    fp, tp = int((ns > thr).sum()), int((os_ > thr).sum())
    tn, fn = int(nvalid.sum()) - fp, int(ovalid.sum()) - tp
    specificity = tn / nvalid.sum() if nvalid.sum() else np.nan
    recall = tp / ovalid.sum() if ovalid.sum() else np.nan
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    f1 = 2*precision*recall/(precision+recall) if precision and recall and (precision+recall) else np.nan
    return dict(split=split_name, rule=rule_name, TN=tn, FP=fp, TP=tp, FN=fn,
                normal_deferred=int((~nvalid).sum()), outlier_deferred=int((~ovalid).sum()),
                normal_detection_pct=100*specificity, false_alarm_pct=100*(1-specificity),
                anomaly_detection_pct=100*recall, miss_pct=100*(1-recall),
                balanced_acc_pct=50*(specificity+recall), F1=f1)

# 베이스라인은 점수 함수가 아니라 OR 규칙(threshold 조합)이라 직접 계산한다
baseline_rows = []
for split in ['validation', 'test']:
    nt = TABLE['normal'][TABLE['normal'].split == split]
    ot = TABLE['outlier']
    nv, na = baseline_predict(nt)
    ov, oa = baseline_predict(ot)
    fp, tp = int(na.sum()), int(oa.sum())
    tn, fn = int(nv.sum()) - fp, int(ov.sum()) - tp
    specificity = tn / nv.sum() if nv.sum() else np.nan
    recall = tp / ov.sum() if ov.sum() else np.nan
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    f1 = 2*precision*recall/(precision+recall) if precision and recall and (precision+recall) else np.nan
    baseline_rows.append(dict(split=split, rule='baseline(04 규칙, 6개 OR)', TN=tn, FP=fp, TP=tp, FN=fn,
                               normal_deferred=int((~nv).sum()), outlier_deferred=int((~ov).sum()),
                               normal_detection_pct=100*specificity, false_alarm_pct=100*(1-specificity),
                               anomaly_detection_pct=100*recall, miss_pct=100*(1-recall),
                               balanced_acc_pct=50*(specificity+recall), F1=f1))

main_rows = []
for split in ['validation', 'test']:
    nt = TABLE['normal'][TABLE['normal'].split == split]
    ot = TABLE['outlier']
    for name, (fn_, thr) in MODELS_MAIN.items():
        main_rows.append(metrics_row(split, name, nt, ot, fn_, thr, FEATURES_MAIN))

METRICS = pd.DataFrame(baseline_rows + main_rows)
for split in ['validation', 'test']:
    print('\n', split)
    print(METRICS[METRICS.split == split].round(3).to_string(index=False))


 validation
     split                   rule  TN  FP  TP  FN  normal_deferred  outlier_deferred  normal_detection_pct  false_alarm_pct  anomaly_detection_pct  miss_pct  balanced_acc_pct    F1
validation baseline(04 규칙, 6개 OR) 102   6  17   0               12                 4                94.444            5.556                100.000     0.000            97.222 0.850
validation            mahalanobis 108   0  17   0               12                 4               100.000            0.000                100.000     0.000           100.000 1.000
validation       isolation_forest 108   0  13   4               12                 4               100.000            0.000                 76.471    23.529            88.235 0.867
validation          one_class_svm 107   1  17   0               12                 4                99.074            0.926                100.000     0.000            99.537 0.971

 test
split                   rule  TN  FP  TP  FN  normal_deferred  outlier_defe

In [11]:
# 폴백 모델(2<=n<10) 평가 — 정상 자체 train/test 분할 + 전체 이상(n=2~9) 기준
fb_rows = []
nv, ns = score_table(test_fb, maha_fb, FEATURES_FALLBACK, min_n=2)
ov, os_ = score_table(out_fb, maha_fb, FEATURES_FALLBACK, min_n=2)
fp, tp = int((ns > maha_fb_thr).sum()), int((os_ > maha_fb_thr).sum())
tn, fn = int(nv.sum()) - fp, int(ov.sum()) - tp
specificity = tn / nv.sum() if nv.sum() else np.nan
recall = tp / ov.sum() if ov.sum() else np.nan
print(f'[폴백 모델, n=2~9] 정상탐지율={100*specificity:.1f}%  오경보율={100*(1-specificity):.1f}%  '
      f'이상탐지율={100*recall:.1f}%  (정상 test {int(nv.sum())}개, 이상 {int(ov.sum())}개)')

[폴백 모델, n=2~9] 정상탐지율=100.0%  오경보율=0.0%  이상탐지율=100.0%  (정상 test 12개, 이상 4개)


**6-fold 블록 교차검증(신뢰구간)** — 단일 분할 점추정의 불확실성을 보강한다
(시간순 블록, 비교 대상 세 모델 모두에 동일하게 적용).

In [12]:
normal_all = TABLE['normal'][TABLE['normal'].n >= MIN_N].sort_values('t_start').reset_index(drop=True)
Kf = 6
cv_blocks = np.array_split(np.arange(len(normal_all)), Kf)
ovalid_cv = (TABLE['outlier'].n >= MIN_N) & np.isfinite(TABLE['outlier'][FEATURES_MAIN]).all(axis=1)
X_out_cv = TABLE['outlier'].loc[ovalid_cv, FEATURES_MAIN].to_numpy(float)

def fit_fold_models(Xtr):
    cov_f = LedoitWolf().fit(Xtr); mean_f = Xtr.mean(axis=0)
    def maha_f(X, cov=cov_f, m=mean_f):
        d = X - m; return np.sqrt(np.einsum('ij,jk,ik->i', d, cov.precision_, d))
    iso_f = IsolationForest(n_estimators=300, random_state=RANDOM_STATE, contamination='auto').fit(Xtr)
    def iso_score_f(X, model=iso_f): return -model.decision_function(X)
    scaler_f = StandardScaler().fit(Xtr)
    ocsvm_f = OneClassSVM(kernel='rbf', gamma='scale', nu=0.1).fit(scaler_f.transform(Xtr))
    def ocsvm_score_f(X, model=ocsvm_f, sc=scaler_f): return -model.decision_function(sc.transform(X))
    fns = {'mahalanobis': maha_f, 'isolation_forest': iso_score_f, 'one_class_svm': ocsvm_score_f}
    return {name: (fn, np.quantile(fn(Xtr), COVERAGE)) for name, fn in fns.items()}

cv_rows = []
for k in range(Kf):
    te = normal_all.iloc[cv_blocks[k]]
    tr = normal_all.iloc[np.concatenate([cv_blocks[j] for j in range(Kf) if j != k])]
    Xtr = tr[FEATURES_MAIN].to_numpy(float); Xte = te[FEATURES_MAIN].to_numpy(float)
    for name, (fn_, thr) in fit_fold_models(Xtr).items():
        cv_rows.append(dict(fold=k, model=name, fp_rate=(fn_(Xte) > thr).mean(), tp_rate=(fn_(X_out_cv) > thr).mean()))

CV_DF = pd.DataFrame(cv_rows)
cv_summary = CV_DF.groupby('model').agg(fp_mean=('fp_rate', 'mean'), fp_sd=('fp_rate', 'std'),
                                         tp_mean=('tp_rate', 'mean'), tp_sd=('tp_rate', 'std'))
se = cv_summary.fp_sd / np.sqrt(Kf)
cv_summary['fp_ci95_lo'] = (cv_summary.fp_mean - 1.96 * se).clip(lower=0)
cv_summary['fp_ci95_hi'] = cv_summary.fp_mean + 1.96 * se
print((cv_summary[['fp_mean', 'fp_sd', 'fp_ci95_lo', 'fp_ci95_hi', 'tp_mean', 'tp_sd']] * 100).round(2))

                  fp_mean  fp_sd  fp_ci95_lo  fp_ci95_hi  tp_mean  tp_sd
model                                                                   
isolation_forest     0.19   0.46         0.0        0.55     95.1    2.4
mahalanobis          0.19   0.46         0.0        0.55    100.0    0.0
one_class_svm        0.37   0.92         0.0        1.11    100.0    0.0


## 6. 추론 데모 — 길이 기반 자동 분기 + 설명력(D² 분해)

실제 운영을 가정해, 세그먼트 1개를 받아 길이에 따라 올바른 모델로 라우팅하고,
이상으로 판정되면 Mahalanobis의 D² 기여도 분해로 원인 피처까지 반환하는 함수를
만든다(현장 활용방안 — 점검 우선순위 안내, `docs/report/04_field_application.md`).

In [13]:
def maha_contrib_main(x):
    diff = x - mean_main
    return diff * (cov_main.precision_ @ diff)  # 합 = D^2

def predict_segment(row):
    n = row['n']
    if n < 2:
        return dict(seg_id=row['seg_id'], n=n, track='deferred', judgement='판정 보류(계산 불가)')
    if n < MIN_N:
        x = row[FEATURES_FALLBACK].to_numpy(float)
        score = float(maha_fb(x.reshape(1, -1))[0])
        judgement = '이상 후보(작업자 확인 요청)' if score > maha_fb_thr else '정상'
        return dict(seg_id=row['seg_id'], n=n, track='fallback(2피처)', score=round(score, 3),
                    threshold=round(maha_fb_thr, 3), judgement=judgement)
    x = row[FEATURES_MAIN].to_numpy(float)
    score = float(maha_main(x.reshape(1, -1))[0])
    judgement = '이상(즉시 경보)' if score > maha_thr else '정상'
    out = dict(seg_id=row['seg_id'], n=n, track='main(3피처)', score=round(score, 3),
               threshold=round(maha_thr, 3), judgement=judgement)
    if score > maha_thr:
        contrib = maha_contrib_main(x)
        top = np.argsort(-np.abs(contrib))[:2]
        out['주요 원인 피처'] = [f'{FEATURES_MAIN[j]}({contrib[j]:.1f})' for j in top]
    return out

demo_rows = []
for _, row in TABLE['outlier'].sort_values('n').iterrows():
    demo_rows.append(predict_segment(row))
print('추론 데모(이상 데이터 전체, 길이 오름차순):')
print(pd.DataFrame(demo_rows).to_string(index=False))

추론 데모(이상 데이터 전체, 길이 오름차순):
 seg_id  n         track   score  threshold        judgement                                              주요 원인 피처
      5  3 fallback(2피처)  26.653      3.794 이상 후보(작업자 확인 요청)                                                   NaN
      0  4 fallback(2피처)  22.412      3.794 이상 후보(작업자 확인 요청)                                                   NaN
     10  4 fallback(2피처)  15.737      3.794 이상 후보(작업자 확인 요청)                                                   NaN
      9  8 fallback(2피처)  13.483      3.794 이상 후보(작업자 확인 요청)                                                   NaN
     19 10     main(3피처) 112.059      5.421        이상(즉시 경보)    [normal_center_nrmse(12552.9), cur_ai1_resid(2.3)]
     14 15     main(3피처)  77.189      5.421        이상(즉시 경보)   [normal_center_nrmse(5752.7), cur_ai0_resid(156.1)]
     20 15     main(3피처) 296.314      5.421        이상(즉시 경보) [normal_center_nrmse(88026.1), cur_ai1_resid(-214.6)]
      4 18     main(3피처)  97.585      5.421        이상

## 7. 최종 결과 요약

In [14]:
print('=== 최종 채택 모델: Mahalanobis (2-track) ===')
print(f'  메인(n>=10, 3피처): 임계={maha_thr:.3f}')
print(f'  폴백(2<=n<10, 2피처): 임계={maha_fb_thr:.3f}')
print(f'  n=1: 판정 보류({len(nrm_defer)}개 정상, {len(out_defer)}개 이상)')
print()
print('=== test 성능(메인 모델, n>=10) ===')
print(METRICS[(METRICS.split == 'test')][['rule', 'normal_detection_pct', 'false_alarm_pct',
                                           'anomaly_detection_pct', 'balanced_acc_pct', 'F1']].round(3).to_string(index=False))
print()
print('=== 6-fold 교차검증 신뢰구간(메인 모델) ===')
print((cv_summary[['fp_mean', 'fp_ci95_lo', 'fp_ci95_hi', 'tp_mean']] * 100).round(2))
print()
print('상세 근거: docs/report/01~06, docs/01_eda_findings.md ~ docs/04_methodology.md, notebooks/01~06 참고.')

=== 최종 채택 모델: Mahalanobis (2-track) ===
  메인(n>=10, 3피처): 임계=5.421
  폴백(2<=n<10, 2피처): 임계=3.794
  n=1: 판정 보류(9개 정상, 0개 이상)

=== test 성능(메인 모델, n>=10) ===
                  rule  normal_detection_pct  false_alarm_pct  anomaly_detection_pct  balanced_acc_pct    F1
baseline(04 규칙, 6개 OR)                80.734           19.266                100.000            90.367 0.618
           mahalanobis               100.000            0.000                100.000           100.000 1.000
      isolation_forest               100.000            0.000                 76.471            88.235 0.867
         one_class_svm               100.000            0.000                100.000           100.000 1.000

=== 6-fold 교차검증 신뢰구간(메인 모델) ===
                  fp_mean  fp_ci95_lo  fp_ci95_hi  tp_mean
model                                                     
isolation_forest     0.19         0.0        0.55     95.1
mahalanobis          0.19         0.0        0.55    100.0
one_class_svm        0.37       